# Survival Simulator — baseline runner

Execution trigger only: all logic lives in `agents/` and `training/` (.py files).

* **Scored runs** (logged to `results/index.csv`) are launched as a **fresh subprocess** via `!python -m training.evaluate`, so kernel state can never leak into a result.
* **Rendering** runs in-kernel and is for watching the game only; its scores are never logged.

Run this notebook with its working directory = `survival-simulator/`.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os, sys, subprocess
# Must run from survival-simulator/ so `src`, `agents`, `training` import
if os.path.basename(os.getcwd()) != "survival-simulator":
    raise RuntimeError(f"cwd is {os.getcwd()} - open the notebook from survival-simulator/")
sys.path.insert(0, os.getcwd())
print(sys.executable)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

## 0. One-time install (skip if already installed)

In [ ]:
# !{sys.executable} -m pip install -r requirements.txt -r requirements-dev.txt

## 1. Tests (run before every scored run)

In [ ]:
!{sys.executable} -m pytest -q tests

## 2. Scored evaluation (fresh process)

Parameters to set per experiment: experiment id, config, seeds, workers (cluster: 4 CPUs), per-game wall-clock cap.
Commit before running — the run records the git revision and flags `-dirty` trees.

In [ ]:
EXPERIMENT = "C1-E00"
CONFIG = "training/configs/dummy_v0.json"
SEEDS = "0 1 2 3 4"
WORKERS = 4
MAX_WALL_SEC = 1800

!{sys.executable} -m training.evaluate --experiment {EXPERIMENT} --config {CONFIG} --seeds {SEEDS} --workers {WORKERS} --max-wall-sec {MAX_WALL_SEC}

In [ ]:
EXPERIMENT = "C1-E01"
CONFIG = "training/configs/heuristic_v0.json"

!{sys.executable} -m training.evaluate --experiment {EXPERIMENT} --config {CONFIG} --seeds {SEEDS} --workers {WORKERS} --max-wall-sec {MAX_WALL_SEC}

## 3. Results index

In [ ]:
import pandas as pd
pd.read_csv("results/index.csv")

## 4. Watch a game (inspection only, not scored)

`max_sim_time` limits how much of the game is rendered (one frame per simulated second by default).

In [ ]:
from IPython.display import Image, display
from training.render import render_episode

gif_path, stats = render_episode("training/configs/heuristic_v0.json", seed=0, max_sim_time=300, frame_every=10, width=640, fps=20)
display(Image(filename=gif_path))
stats